In [1]:
# ============================================================
# CELL 0 – RUN THIS FIRST IN EVERY NOTEBOOK (do not modify)
# ============================================================
import os, sys
from google.colab import drive
import torch

# Step 1: Mount Google Drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = "/content/drive/MyDrive/HateSpeech_NLP"

# Step 2: Clone or pull the Git repository
# CHANGE THIS TO YOUR ACTUAL REPO URL
REPO_URL  = "https://github.com/thong7d/hate-speech-detection.git"
REPO_NAME = "hate-speech-detection"
REPO_PATH = f"/content/{REPO_NAME}"

if not os.path.exists(REPO_PATH):
    os.system(f"git clone {REPO_URL} {REPO_PATH}")
else:
    os.system(f"cd {REPO_PATH} && git pull origin main")

# Step 3: Install the package in editable mode
os.system(f"pip install -q -e {REPO_PATH}")

# Step 4: Add src/ to Python path
if f"{REPO_PATH}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_PATH}/src")

# Step 5: Install remaining dependencies
os.system(f"pip install -q -r {REPO_PATH}/requirements.txt")

print("✅ Environment ready. REPO_PATH:", REPO_PATH)

Mounted at /content/drive
✅ Environment ready. REPO_PATH: /content/hate-speech-detection


In [4]:
# ============================================================
# CELL 1 – TRAIN BASELINE MODEL (ROBUST PATH HANDLING)
# ============================================================
import os
import pandas as pd
import yaml
import json
import joblib
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

# 1. Load configuration
with open(f"{REPO_PATH}/configs/paths.yaml", 'r') as f:
    paths = yaml.safe_load(f)

def ensure_string_path(path_obj, default_val):
    """
    Trích xuất chuỗi đường dẫn nếu path_obj là dictionary,
    ngược lại trả về chính nó hoặc giá trị mặc định.
    """
    if isinstance(path_obj, dict):
        # Ưu tiên lấy key 'dir' hoặc giá trị đầu tiên trong dict
        return path_obj.get('dir', list(path_obj.values())[0])
    return path_obj if isinstance(path_obj, str) else default_val

# Xử lý an toàn các đường dẫn từ YAML
raw_data_sub = ensure_string_path(paths.get('raw_data'), 'data/raw/vihsd')
processed_sub = ensure_string_path(paths.get('processed_data'), 'data/processed')
models_sub = ensure_string_path(paths.get('models'), 'models')
results_sub = ensure_string_path(paths.get('results'), 'results')

# Thiết lập đường dẫn tuyệt đối dựa trên DRIVE_ROOT
processed_dir = os.path.join(DRIVE_ROOT, processed_sub)
model_dir = os.path.join(DRIVE_ROOT, models_sub, 'baseline')
results_dir = os.path.join(DRIVE_ROOT, results_sub)

# 2. Load data
train = pd.read_parquet(os.path.join(processed_dir, "train.parquet"))
dev = pd.read_parquet(os.path.join(processed_dir, "dev.parquet"))
print(f"✅ Data loaded from: {processed_dir}")

# 3. Feature Extraction (Char n-grams 2-5)
tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 5),
    max_features=50000,
    sublinear_tf=True
)
X_train = tfidf.fit_transform(train['text'])
X_dev = tfidf.transform(dev['text'])

# 4. Train Model
# Tự động xử lý mất cân bằng nhãn bằng class_weight='balanced'
model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, train['label_id'])

# 5. Evaluate
y_pred = model.predict(X_dev)
macro_f1 = f1_score(dev['label_id'], y_pred, average='macro')
print(f"\n🚀 Baseline Macro F1 Score: {macro_f1:.4f}")
print("\nClassification Report:\n", classification_report(dev['label_id'], y_pred, target_names=['CLEAN', 'OFFENSIVE', 'HATE']))

# 6. Save Artifacts
os.makedirs(model_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

joblib.dump(model, os.path.join(model_dir, "tfidf_lr.pkl"))
joblib.dump(tfidf, os.path.join(model_dir, "tfidf_vectorizer.pkl"))

# Lưu báo cáo kết quả
with open(os.path.join(results_dir, "baseline_report.json"), 'w') as f:
    json.dump({"macro_f1": float(macro_f1)}, f, indent=4)

print(f"✅ Baseline artifacts saved successfully to Drive.")

✅ Data loaded from: /content/drive/MyDrive/HateSpeech_NLP/data/processed

🚀 Baseline Macro F1 Score: 0.6129

Classification Report:
               precision    recall  f1-score   support

       CLEAN       0.96      0.84      0.89      2180
   OFFENSIVE       0.36      0.52      0.43       212
        HATE       0.42      0.69      0.52       270

    accuracy                           0.80      2662
   macro avg       0.58      0.68      0.61      2662
weighted avg       0.86      0.80      0.82      2662

✅ Baseline artifacts saved successfully to Drive.
